## **Predictive Analysis of Inventory Dead stock**

### Objective: 
- This assignment focuses on a dataset of Inventory dead stock.
- The primary goal is to predict which stock items are likely to be dead (e.g. not used or moved in a long time) using various data mining methods.

### Background: 
A mid manufacturing retailer noticed that certain items in its warehouse across Australia had not moved for a number of months. These products included a number of products or materials stored or manufactured in the wharehouses. Despite being listed in the inventory system, they had no recent sales activity, were no longer featured in marketing campaigns, and had limited relevance to current customer needs.

The inventory manager initiated a review to determine whether these items should be classified as dead stock—inventory that is unlikely to be sold due to obsolescence, lack of demand, damage, or other reasons that can impact their classification. The classification would allow the company to make informed decisions about markdowns, liquidation, or disposal, and free up valuable warehouse space. This project aims to refine this classification by identifying stock items (and possibly their wharehouses locations) that are likely to be dead stock, enhancing the precision of wharhouse facility storage planning

Target variable
- Dead stock (1=yes, 0=no)

The dataset contains the features listed in the second **"Readme"** sheet of the Dataset file





---

**Problem 1** - Reading the dataset 



**Q1**. Create a pandas dataframe contining the first 3,000 rows from the Inventory dataset provided in the **Dataset** folder 
- Delete 'Item No.' column
- Print `info()` of the dataframe



- Do you recommend dropping/deleting any another column(s) from the Inventory dataset? 

If **YES** then:
1- Identify which one(s) and
2- Justify your selection in one sentence

If **NO** then Justify your answer





In [1]:
import pandas as pd

# Q1 (5 marks)
# 1. Create a pandas DataFrame named `df` containing only the **first 3,000 rows** from `Inventory.csv`.
# Note: The provided snippet seems to be truncated or a sample. Assuming the full file is available.
df = pd.read_csv('Dataset/Inventory.csv', nrows=3000, encoding='latin1')

# 2. Delete the `'Item No.'` column from the DataFrame.
df.drop(columns=['Item No. '], inplace=True) # Note the space after 'No.' as per the provided snippet, in the csv or xlsx file are the same

# 3. Print the `df.info()` summary.
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 32 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Item Description                  2999 non-null   object 
 1   Whse                              3000 non-null   object 
 2   State                             3000 non-null   object 
 3   Base Unit                         3000 non-null   object 
 4   Total - Quantity                  3000 non-null   float64
 5   Inventory Aging Report Unit Cost  3000 non-null   float64
 6   Total - Value                     3000 non-null   float64
 7   6 Months QTY                      3000 non-null   float64
 8   12 Months QTY                     3000 non-null   float64
 9   2 Years QTY                       3000 non-null   float64
 10  Over 2 Years Qty                  3000 non-null   float64
 11  Over 3 Years Quantity             3000 non-null   float64
 12  Busine

Based on the data dictionary and `df.info()` output, do you recommend dropping any other columns?

**YES**

- **`'Item Description'`**: This is a text description. For a standard predictive model, it's typically dropped unless specific text features are engineered.
- **`'Over 2 years Qty'`**: We have the Over 3 Years Qty, which is a superset of this. Including both could introduce redundancy.

**Q2**. List which **features** are *numeric*, *ordinal*, and *nominal* variables, and how many features of each kind there are in the dataset.
To answer this question 

- Find the definitions of numeric, ordinal and nominal variables in the course material    
- Carefully consider what values each feature can take as well as the output of `df.info()`.
- Make sure to check the **Readme** sheet in the dataset to understand the features description  

Your answer should be written up in Markdown and include:
1) A table listing all the features present in the dataset and their type (fill out the table template provided below) and
2) A brief description of the contents of the table.

|Variable Kind|Number of Features|Feature Names
| --- | --- | --- |
| Numeric | some number | some text |
| some text  | some number | some text |
| some text  | some number | some text |

Answer table:

| Variable Kind | Number of Features | Feature Names |
| --- | --- | --- |
| Numeric | 25 | "Total - Quantity", "Inventory Aging Report Unit Cost", "Total - Value", "6 Months QTY", "12 Months QTY", "2 Years QTY", "Over 2 Years Qty", "Over 3 Years Quantity", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Avg monthly", "Inventory Turn", "% of over 2 year", "Over 2 years Qty" |
| Ordinal | 1 | "ABC Class" (A > B > C > D > E > F > H > I > J, though not strictly numerical, it represents a ranking/importance) |
| Nominal | 7 | "Item Description", "Whse", "State", "Base Unit", "Business Area", "Item Type", "Dead stock" |


This table categorizes the features in the inventory dataset. Numeric features represent measurable quantities or continuous values. The **ABC Class** is treated as ordinal due to its inherent ranking system (A being most important). The remaining categorical features are nominal as they represent distinct categories without a natural order.

**Q3.** Missing Values. 

- Print out the number of missing values for each variable in the dataset and comment on your findings.

In [ ]:
# Code:
# 1. Calculate and print the number of missing values for each variable in the `df` DataFrame.
df = pd.read_csv('Dataset/Inventory.csv', encoding='latin1')
print(df.isnull().sum())

Item No.                            0
Item Description                    1
Whse                                0
State                               0
Base Unit                           0
Total - Quantity                    0
Inventory Aging Report Unit Cost    0
Total - Value                       0
6 Months QTY                        0
12 Months QTY                       0
2 Years QTY                         0
Over 2 Years Qty                    0
Over 3 Years Quantity               0
Business Area                       0
Item Type                           0
ABC Class                           0
Jan                                 0
Feb                                 0
Mar                                 0
Apr                                 0
May                                 0
Jun                                 0
Jul                                 0
Aug                                 0
Sep                                 0
Oct                                 0
Nov         

Briefly comment on your findings regarding the missing values.

*   The output shows that the `'Item Description'` column has 1 missing value. 
*   All other columns in the DataFrame appear to have 0 missing values. 
*   This is a perfect dataset with respect to the amount of missing data. 
*   So we can concentrate on a column that is likely to be dropped for modeling anyway. 
*   This makes data cleaning straightforward for the rest of the features.

**Problem 2.** Cleaning data and dealing with categorical features


**Q1.** 
- Use an appropriate `pandas` function to impute missing values using one of the following two strategies: `mean` and `mode`.
    - Take into consideration the type of each variable (as in Q2 above) and the best practices we discussed in class/lecture notes
- Explain what data imputation is, how you have done it here, and what decisions you had to make.

In [3]:
# Problem 2: Cleaning data and dealing with categorical features (40 Marks)

# Q1 (15 marks)
# Code:
# 1. Impute all missing values in the DataFrame `df`.
# 2. Use the **mean** imputation strategy for numeric variables.
# 3. Use the **mode** imputation strategy for categorical (nominal and ordinal) variables.

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
# Excluding 'Dead stock' from imputation as it's the target variable
if 'Dead stock' in numeric_cols:
    numeric_cols.remove('Dead stock')

# For simplicity in this context, we'll treat 'ABC Class' as categorical for mode imputation
# as it's a label, even though it's ordinal.
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
# Ensure 'ABC Class' is in categorical if it wasn't picked up (it might be object dtype)
if 'ABC Class' not in categorical_cols and 'ABC Class' in df.columns:
    categorical_cols.append('ABC Class')
    if 'ABC Class' in numeric_cols:
        numeric_cols.remove('ABC Class')

# Impute numeric columns with mean
for col in numeric_cols:
    df[col].fillna(df[col].mean(), inplace=True)

# Impute categorical columns with mode
for col in categorical_cols:
    # Mode can return multiple values, so we take the first one
    mode_value = df[col].mode()
    if not mode_value.empty:
        df[col].fillna(mode_value[0], inplace=True)
    else:
        # If mode is empty (e.g., all values are NaN), fill with a placeholder
        df[col].fillna('Unknown', inplace=True)

print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
Item No.                            0
Item Description                    0
Whse                                0
State                               0
Base Unit                           0
Total - Quantity                    0
Inventory Aging Report Unit Cost    0
Total - Value                       0
6 Months QTY                        0
12 Months QTY                       0
2 Years QTY                         0
Over 2 Years Qty                    0
Over 3 Years Quantity               0
Business Area                       0
Item Type                           0
ABC Class                           0
Jan                                 0
Feb                                 0
Mar                                 0
Apr                                 0
May                                 0
Jun                                 0
Jul                                 0
Aug                                 0
Sep                                 0
Oct              

C:\Users\Admin\AppData\Local\Temp\ipykernel_21212\2709567297.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_21212\2709567297.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

Explain what data imputation is.
Describe how you performed it and justify your choice of mean for numeric and mode for categorical features.

**Data Imputation** is the process of replacing missing data with substituted values to maintain the integrity and usability of a dataset for analysis.

I performed imputation by first identifying numeric and categorical columns:
*   For numeric variables (e.g., `Total - Quantity`, `Avg monthly`), I used **mean imputation**, replacing missing values with the average value of that column => This is a common strategy for numeric data as it preserves the overall mean of the dataset. 
*   For categorical variables (e.g., `State`, `ABC Class`), I used **mode imputation**, replacing missing values with the most frequently occurring category in that column => This is suitable for categorical data as it uses the most "typical" category, avoiding the creation of artificial or meaningless values.

**Q2**. 
- Print `value_counts()` of the 'State' column and add a dummy variable named 'State_NSW' to `df` using `get_dummies()` (3 marks)
- Carefully explain what the values of the new variable 'State_NSW' mean (2 mark)
- Make sure the variable 'State' is deleted from the original dataframe   

In [4]:
# Q2 (5 marks)
# Code:
# 1. Print the `value_counts()` for the `'State'` column.
print(df['State'].value_counts())

# 2. Create a new dummy variable column named `'State_NSW'` in `df` using `pd.get_dummies()` on the `'State'` column.
# Using drop_first=False to explicitly create the column we want
state_dummies = pd.get_dummies(df['State'], prefix='State', drop_first=False)
# Select only the 'State_NSW' column
df['State_NSW'] = state_dummies['State_NSW']

# 3. Delete the original `'State'` column from `df`.
df.drop(columns=['State'], inplace=True)

# Display the new column
print(df[['State_NSW']].head())

State
NSW    2423
WA     1029
Name: count, dtype: int64
   State_NSW
0       True
1       True
2      False
3       True
4       True


Carefully explain what the values (0 and 1) in the new `'State_NSW'` column represent.

The new column `'State_NSW'` is a binary (dummy) variable created from the original `'State'` column.
- A value of **1** in `'State_NSW'` indicates that the corresponding inventory item is located in a warehouse in the state of **New South Wales (NSW)**.
- A value of **0** indicates that the item is located in a warehouse in a **different state** (in this case, Western Australia 'WA', based on the `value_counts()` output).

**Q3**. Print `value_counts()` of the 'Whse' column and *carefully* comment on what you notice in relation to the definition of this variable. 

In [5]:
# Q3 (5 marks)
# Code:
# 1. Print the `value_counts()` of the `'Whse'` column.
print(df['Whse'].value_counts())

Whse
1N1    1574
1W0    1029
1N0     849
Name: count, dtype: int64


**Q4**. 

- Apply `get_dummies()` to 'Whse' feature and add dummy variables 'Whse_1N0', 'Whse_1N1', 'Whse_1W0' to `df`. 
- *Carefully consider* how to allocate all the values of 'Whse' across these 3 newly created features 
    - Do not delete observations 
    - Do not assume that the anomaly are missing observations      
    - Explain what decision you made
- Make sure that 'Whse' is deleted from `df`    

In [ ]:
# 1. Apply `pd.get_dummies()` to the `'Whse'` feature to create three new dummy variables: `'Whse_1N0'`, `'Whse_1N1'`, and `'Whse_1W0'`.
# 2. Ensure all original values from `'Whse'` are correctly allocated across these new columns.
whse_dummies = pd.get_dummies(df['Whse'], prefix='Whse')

# Add the dummy columns to the dataframe
df = pd.concat([df, whse_dummies], axis=1)

# 3. Delete the original `'Whse'` column from `df`.
df.drop(columns=['Whse'], inplace=True)

# Verify the creation
print(df[['Whse_1N0', 'Whse_1N1', 'Whse_1W0']].head())
print(f"\nShape of dummy columns: {df[['Whse_1N0', 'Whse_1N1', 'Whse_1W0']].shape}")
print(f"Unique values in original 'Whse': {df.filter(regex='Whse_').columns.tolist()}")

   Whse_1N0  Whse_1N1  Whse_1W0
0      True     False     False
1     False      True     False
2     False     False      True
3     False      True     False
4     False      True     False

Shape of dummy columns: (3452, 3)
Unique values in original 'Whse': ['Whse_1N0', 'Whse_1N1', 'Whse_1W0']


Explain the decisions you made during this one-hot encoding process.

*   I used `pd.get_dummies()` on the `'Whse'` column. This function automatically creates a new binary column for each unique category found in `'Whse'` (in this case, `1N0`, `1N1`, `1W0`). 
*   I chose not to use `drop_first=True` to keep all categories explicitly, which can sometimes be clearer for interpretation. 
*   After creating the dummy variables, I concatenated them back to the main DataFrame and then removed the original `'Whse'` column to avoid redundancy, as its information is now fully represented by the new dummy variables.

<hr style="width:25%;margin-left:0;"> 

**Q5**. In the column 'Item Type', convert the values of all items that are FG --- into "Finished Goods", RM --- into "Raw Materials", keep "WIP Manufactured", while combine all others like [Subcontract (9), Customer Supplied (08), and  zItems] into type 'Other' so you have Four Item types **Finished Goods**, **Raw Materials**, **WIP Manufactured**, and **Other**. 

Now Encode all the remaining non numeric features using appropriate encoding method presented in class

In [ ]:
# Code:
# 1. The `'Item Type'` column contains values like `'RM Purchased Local (4)'`. Create a new feature called `'Item_Category'` that extracts only the abbreviation (e.g., 'RM').
# 2. Create another new feature called `'Source'` that extracts the source description (e.g., 'Purchased Local').
# 3. Delete the original `'Item Type'` column.

import re

# Example value: 'RM Purchased Local (4)'
# Pattern: Capture letters at the start, then space, then letters/space until ' ('
df['Item_Category'] = df['Item Type'].str.extract(r'^([A-Z]+)')
# This pattern captures the part before the first space reliably for the category abbreviation.
# For Source, we need the part after the category and before the number in parentheses.
# Let's refine the extraction for Source to be more robust.
def extract_source(item_type_str):
    # Match everything after the initial letters and space, up to but not including ' ('
    match = re.search(r'^[A-Z]+\s+(.+?)\s*\(\d+\)$', item_type_str)
    if match:
        return match.group(1)
    else:
        # Fallback if format is unexpected
        return "Unknown_Source"

df['Source'] = df['Item Type'].apply(extract_source)

# 3. Delete the original `'Item Type'` column.
df.drop(columns=['Item Type'], inplace=True)

# Display results
print(df[['Item_Category', 'Source']].head(10))
print("\nValue counts for 'Item_Category':")
print(df['Item_Category'].value_counts())
print("\nValue counts for 'Source':")
print(df['Source'].value_counts())

  Item_Category            Source
0            RM   Purchased Local
1            FG    Sourced Import
2            FG    Sourced Import
3            FG    Sourced Import
4            FG    Sourced Import
5            FG    Sourced Import
6            FG    Sourced Import
7            RM  Purchased Import
8            RM  Purchased Import
9            RM  Purchased Import

Value counts for 'Item_Category':
Item_Category
FG     2405
RM      793
WIP     227
C        18
S         8
Name: count, dtype: int64

Value counts for 'Source':
Source
Sourced Import                    845
Purchased Import                  764
Purchased Local                   591
Purchased Philips Professional    381
Purchased Philips Conventional    335
Manufactured                      275
Purchased Philips Consumer        230
Unknown_Source                     27
Purchased Philips OEM               4
Name: count, dtype: int64


Explain the steps you took to engineer these two new features.

To create `'Item_Category'`, I used `str.extract()` with a regular expression `r'^([A-Z]+)'`. This pattern captures one or more uppercase letters (`[A-Z]+`) that appear at the very beginning (`^`) of the string, effectively isolating the abbreviation like 'RM' or 'FG'.

To create `'Source'`, I defined a custom function `extract_source`. This function uses a more complex regular expression `r'^[A-Z]+\s+(.+?)\s*$'` to find the text that comes after the initial category abbreviation and space, capturing everything (`(.+?)`) up to the number in parentheses at the end (`\s*$`). This reliably extracts descriptions like 'Purchased Local' or 'Sourced Import'. After successfully creating these two new, more granular features, the original, combined `'Item Type'` column was deleted.

**Problem 3** Preparing X and y arrays

**Q1**. 

- Create a numpy array `y` from the first 80% observations of `Dead stock` column from `df` 
- Create a numpy array `X`  from the first 80% observations of all the remaining variables in `df` 

In [8]:
# Q1
# Based on Q1, we also recommended dropping 'Item Description'
columns_to_drop = ['Item Description', 'Dead stock']
# Ensure 'Over 2 years Qty' is also dropped if it exists (duplicate)
if 'Over 2 years Qty' in df.columns:
    columns_to_drop.append('Over 2 years Qty')

X = df.drop(columns=columns_to_drop)
y = df['Dead stock']

# Encode target vector y
y_encoded = y.map({'Yes': 1, 'No': 0})

# One-hot encode all remaining categorical features in X
# Identify categorical columns in X
categorical_cols_X = X.select_dtypes(include=['object']).columns.tolist()
# print("Categorical columns to encode:", categorical_cols_X) # For debugging

# Apply one-hot encoding
X_encoded = pd.get_dummies(X, columns=categorical_cols_X, drop_first=True)

**Q2**. 

- Use an appropriate `sklearn` library we used in class to create `y_train`, `y_test`, `X_train` and `X_test` by splitting the data into 80% train and 20% test datasets 
    - Set random_state to 31 and stratify the subsamples so that train and test datasets have roughly equal proportions of the target's class labels 
- Standardise the data to mean zero and variance one using an approapriate `sklearn` library

In [9]:
# Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

# Scale the numeric features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit scaler on training data and transform both train and test
# It's crucial to only fit on training data to avoid data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


**Problem 4**. Training Models and Interpretation

**Q1**. 

- Train one linear classifier we studied in class using standardised data 
- Compute and print training and test dataset accuracies

In [25]:
# Train 1 linear model (Logistic Regression)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score
linear_model = LogisticRegression(max_iter=50, random_state=31) # Increased max_iter to ensure convergence
linear_model.fit(X_train_scaled, y_train)

# Compute and print training and test dataset accuracies (4 marks)
y_train_pred_linear = linear_model.predict(X_train_scaled)
y_test_pred_linear = linear_model.predict(X_test_scaled)

train_acc_linear = accuracy_score(y_train, y_train_pred_linear)
test_acc_linear = accuracy_score(y_test, y_test_pred_linear)

print("--- Linear Classifier (Logistic Regression) ---")
print(f"Training Accuracy: {train_acc_linear:.4f}")
print("These metrics are stated in the assignment, but I use it to evaluate regression models. Making sure that it not overfitting the data.")
print(f"Test Accuracy: {test_acc_linear:.4f}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_test_pred_linear):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_test_pred_linear):.4f}")
print(f"R^2 Score: {r2_score(y_test, y_test_pred_linear):.4f}")

--- Linear Classifier (Logistic Regression) ---
Training Accuracy: 0.9986
These metrics are stated in the assignment, but I use it to evaluate regression models. Making sure that it not overfitting the data.
Test Accuracy: 0.9957
Mean Absolute Error: 0.0043
Mean Squared Error: 0.0043
R^2 Score: 0.9256


**Q2.**
- Train one nonlinear classifier we studied in class on the same dataset 
- Compute and print training and test dataset accuracies 

In [ ]:
# Train one nonlinear classifier we studied in class on the same dataset (6 marks)
# We'll use Random Forest as our nonlinear classifier
# Note: Tree-based models like Random Forest do not require feature scaling
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
nonlinear_model = RandomForestClassifier(n_estimators=1, random_state=31) # Using 1 tree for demonstration
nonlinear_model.fit(X_train, y_train) # Using unscaled data for Random Forest

# Compute and print training and test dataset accuracies (4 marks)
y_train_pred_nonlinear = nonlinear_model.predict(X_train)
y_test_pred_nonlinear = nonlinear_model.predict(X_test)

train_acc_nonlinear = accuracy_score(y_train, y_train_pred_nonlinear)
test_acc_nonlinear = accuracy_score(y_test, y_test_pred_nonlinear)

print("--- Nonlinear Classifier (Random Forest) ---")
print(f"Training Accuracy: {train_acc_nonlinear:.4f}")
print(f"Test Accuracy: {test_acc_nonlinear:.4f}")
print("These metrics are stated in the assignment, but I use it to evaluate regression models. Making sure that it not overfitting the data.")
print(f"Precision Score: {precision_score(y_test, y_test_pred_nonlinear):.4f}")
print(f"Recall Score: {recall_score(y_test, y_test_pred_nonlinear):.4f}")
print(f"F1 Score: {f1_score(y_test, y_test_pred_nonlinear):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_test_pred_nonlinear)}")

--- Nonlinear Classifier (Random Forest) ---
Training Accuracy: 0.9917
Test Accuracy: 0.9841
Precision Score: 0.8636
Recall Score: 0.8837
F1 Score: 0.8736
Confusion Matrix:
[[642   6]
 [  5  38]]



**Q3**. 

- Comment on the accuracy results obtained from the two classifiers 
- Based on our investigation into Dead stock predictions in this assignment, which model would you recommend and why? 
- What **feature(s)** do you believe has/have higest impact on predicting the outcome variable? Why?


**Comment on the accuracy results obtained from the two classifiers (6 marks)**

-   The Logistic Regression model achieved very high accuracy on both training (0.9986) and test (0.9957) sets, with a small gap between them, suggesting it generalizes well without significant overfitting. 
-   The Random Forest model also performed well, with high training (0.9917) and test (0.9841) accuracy, showing a slightly larger gap, which might indicate a bit more overfitting to the training data compared to Logistic Regression. However, both models demonstrate excellent predictive performance on this dataset.

**Based on our investigation into Dead stock predictions in this assignment, which model would you recommend and why? (4 marks)**

-   I would recommend the **Logistic Regression** model. While both models have high accuracy, Logistic Regression shows slightly better generalization (smaller train-test accuracy gap) and is simpler, more interpretable, and computationally efficient. 
-   For a critical business prediction like dead stock, a simpler, robust model with clear coefficients can be preferable for understanding feature impact and making reliable predictions.

**What **feature(s)** do you believe has/have the highest impact on predicting the outcome variable? Why?**

-   Features related to **inventory aging** likely have the highest impact, particularly `'Over 2 Years Qty'`, `'% of over 2 year'`, and `'Over 3 Years Quantity'`. 
-   These directly measure how long stock has been unsold or unused. 
-   Items that have been in inventory for over 2 or 3 years are strong indicators of dead stock, as defined by the target variable. 
-   The high accuracy of the models suggests these age-related features are powerful predictors in distinguishing between dead and active stock.